# Iris Classification Pipeline — Mage Companion Notebook

**Student:** Aayushi Rajesh Thakre  
**Roll No.:** 23BCS004

This notebook presents a reproducible, cell-by-cell view of the Iris ML workflow implemented as Mage blocks in `iris_project/`. The Mage orchestration graph, decorators, block tests, and CLI execution remain in the Mage project. This notebook mirrors the data and model steps so their intermediate outputs and evaluation are inspectable in notebook form.

**Pipeline:** `load_iris → validate_data → extra_quality_check → prepare_data → train_model → (evaluate_model, save_results)`


## 1. Imports and configuration

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RANDOM_STATE = 42
TEST_SIZE = 0.20
MAX_DEPTH = 3
MINIMUM_ACCURACY = 0.70


## 2. Load Iris data (Mage: `load_iris`)

In [2]:
dataset = load_iris(as_frame=True)
df = dataset.frame.copy()
df["species"] = df["target"].map(dict(enumerate(dataset.target_names)))
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

Rows: 150
Columns: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target', 'species']


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,species
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


## 3. Validate and run the additional quality check

In [3]:
assert not df.isnull().any().any(), "Iris data contains missing values"
assert df["target"].nunique() == 3, "Expected exactly three classes"

required = {
    "sepal length (cm)", "sepal width (cm)",
    "petal length (cm)", "petal width (cm)", "target"
}
assert len(df) > 0, "Iris dataset is empty"
missing = required.difference(df.columns)
assert not missing, f"Missing required columns: {sorted(missing)}"
assert not df[list(required)].isnull().any().any(), "Required columns contain nulls"
assert df["target"].nunique() == 3, "Expected exactly three Iris classes"

print("Validation passed: non-empty data, required columns present, no nulls, three classes.")

Validation passed: non-empty data, required columns present, no nulls, three classes.


## 4. Prepare stratified 80:20 split (Mage: `prepare_data`)

In [4]:
train_idx, test_idx = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["target"],
)
prepared = df.copy()
prepared["split"] = "train"
prepared.loc[test_idx, "split"] = "test"
train_df = prepared[prepared["split"] == "train"].copy()
test_df = prepared[prepared["split"] == "test"].copy()

assert len(train_df) == 120
assert len(test_df) == 30
print(f"Training rows: {len(train_df)}")
print(f"Testing rows: {len(test_df)}")
print("Training class counts:")
print(train_df["species"].value_counts().sort_index())
print("Testing class counts:")
print(test_df["species"].value_counts().sort_index())

Training rows: 120
Testing rows: 30
Training class counts:
species
setosa        40
versicolor    40
virginica     40
Name: count, dtype: int64
Testing class counts:
species
setosa        10
versicolor    10
virginica     10
Name: count, dtype: int64


## 5. Train decision tree and apply quality gate (Mage: `train_model`)

In [5]:
feature_columns = [
    "sepal length (cm)", "sepal width (cm)",
    "petal length (cm)", "petal width (cm)"
]
model = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=RANDOM_STATE)
model.fit(train_df[feature_columns], train_df["target"])
y_true = test_df["target"].to_numpy()
y_pred = model.predict(test_df[feature_columns])
accuracy = float(accuracy_score(y_true, y_pred))
print(f"max_depth: {MAX_DEPTH}")
print(f"Test accuracy: {accuracy:.4f} ({accuracy:.2%})")
assert accuracy >= MINIMUM_ACCURACY, (
    f"Accuracy {accuracy:.3f} is below the {MINIMUM_ACCURACY:.2f} quality threshold; "
    "downstream steps must not run."
)
print("Quality gate passed.")

max_depth: 3
Test accuracy: 0.9667 (96.67%)
Quality gate passed.


## 6. Evaluate model (Mage: `evaluate_model`)

In [6]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
print("Confusion matrix (rows=actual, columns=predicted):")
print(cm)
print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    labels=[0, 1, 2],
    target_names=list(dataset.target_names),
    zero_division=0,
))

Confusion matrix (rows=actual, columns=predicted):
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Classification report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



## 7. Export predictions (Mage: `save_results`)

In [7]:
predictions = pd.DataFrame({"actual": y_true, "predicted": y_pred})
output_path = Path("iris_predictions.csv")
predictions.to_csv(output_path, index=False)
print(f"Saved {len(predictions)} predictions to {output_path.resolve()}")
predictions.head(10)

Saved 30 predictions to /mnt/data/iris_predictions.csv


,actual,predicted
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0


## 8. Pipeline summary

The Mage project contains the actual orchestrated blocks and block-level tests. This companion notebook reproduces the main data preparation, model training, quality gate, evaluation, and export steps in a linear notebook for review. The authoritative Mage run evidence is the CLI output and Mage UI graph included in the report appendix.